In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q flask flask-ngrok pyngrok twilio openai-whisper groq gtts pydub
!pip install -q beautifulsoup4 requests
!pip install -q sentence-transformers faiss-cpu numpy

!apt-get install -y ffmpeg -q

print("✅ All dependencies installed.")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
✅ All dependencies installed.


In [ ]:
# ============================================================
# CELL 2: Configuration — API Keys & Settings
# ============================================================
import os
from twilio.rest import Client

GROQ_API_KEY        = ""
TWILIO_ACCOUNT_SID  = ""
TWILIO_AUTH_TOKEN   = ""
TWILIO_PHONE_NUMBER = "+14057048906"
NGROK_AUTH_TOKEN    = ""

WHISPER_MODEL = "base"
LLAMA_MODEL   = "llama-3.1-8b-instant"

DOCS_SITE_URL       = "https://document.hivestaff.in/"
MAX_PAGES_TO_SCRAPE = 40

CHUNK_SIZE    = 400
CHUNK_OVERLAP = 80
TOP_K_CHUNKS  = 3        # ← reduced from 5 (faster RAG retrieval)
EMBED_MODEL   = "all-MiniLM-L6-v2"

RECORD_MAX_LENGTH = 45
RECORD_TIMEOUT    = 2    # ← reduced from 3 (faster silence detection)

SPEECH_HINTS = (
    "HiveStaff, employee, schedule, HR, payroll, recruitment, attendance, "
    "timesheet, department, shift, leave, salary, assets, contractor, "
    "warehouse, purchase, materials, reception, leads, notice board, "
    "key objectives, staff management, modules, workflow, CRM"
)

os.environ["GROQ_API_KEY"]       = GROQ_API_KEY
os.environ["TWILIO_ACCOUNT_SID"] = TWILIO_ACCOUNT_SID
os.environ["TWILIO_AUTH_TOKEN"]  = TWILIO_AUTH_TOKEN

twilio_client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)

print("✅ Configuration set.")

✅ Configuration set.


In [3]:
!pip install docling -q

In [4]:
# ============================================================
# CELL 2.5: Build RAG using Docling (PDF mode)
# ============================================================
import os, faiss, numpy as np, logging
logging.getLogger("docling").setLevel(logging.ERROR)
logging.getLogger("pdfminer").setLevel(logging.ERROR)

os.system("pip install docling pymupdf -q")

from docling.document_converter import DocumentConverter
from sentence_transformers import SentenceTransformer

# ── 🔧 Using PDF (your already-generated file) ───────────────
KNOWLEDGE_FILES = [
    "/content/sample_data/hivestaff_docs.pdf"   # ← your uploaded PDF
]

# ── Convert PDF using Docling ────────────────────────────────
converter    = DocumentConverter()
all_raw_text = {}

print(f"📄 Converting {len(KNOWLEDGE_FILES)} files with Docling...\n")

for path in KNOWLEDGE_FILES:
    if not os.path.exists(path):
        print(f"   ❌ File not found: {path}")
        continue
    try:
        print(f"   🔄 {path}")
        result = converter.convert(path)
        md     = result.document.export_to_markdown()

        if md.strip():
            all_raw_text[path] = md
            print(f"      ✅ Docling: {len(md):,} chars")
        else:
            # Fallback to pymupdf if Docling gives empty output
            print(f"      ⚠️  Docling empty — using pymupdf fallback...")
            import fitz
            doc  = fitz.open(path)
            text = "\n".join(p.get_text() for p in doc)
            doc.close()
            if text.strip():
                all_raw_text[path] = text
                print(f"      ✅ pymupdf fallback: {len(text):,} chars")
            else:
                print(f"      ❌ Both methods gave empty output for: {path}")

    except Exception as e:
        print(f"      ❌ Error: {e}")
        # Fallback to pymupdf on any exception
        try:
            import fitz
            doc  = fitz.open(path)
            text = "\n".join(p.get_text() for p in doc)
            doc.close()
            if text.strip():
                all_raw_text[path] = text
                print(f"      ✅ pymupdf fallback: {len(text):,} chars")
        except Exception as e2:
            print(f"      ❌ Fallback also failed: {e2}")

if not all_raw_text:
    raise RuntimeError(
        "❌ No content loaded.\n"
        "   Make sure hivestaff_docs.pdf is uploaded to Colab at:\n"
        "   /content/sample_data/hivestaff_docs.pdf"
    )

# ── Save markdown for inspection ─────────────────────────────
os.makedirs("/content/docs_markdown", exist_ok=True)
for i, (source, md) in enumerate(all_raw_text.items()):
    fname = f"/content/docs_markdown/doc_{i+1:02d}.md"
    with open(fname, "w") as f:
        f.write(f"# Source: {source}\n\n{md}")
print(f"\n💾 Markdown saved to /content/docs_markdown/ (inspect if needed)")

# ── Chunk text ───────────────────────────────────────────────
def chunk_text(text, source, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start  = 0
    while start < len(text):
        end   = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append({"text": chunk, "source": source})
        start += chunk_size - overlap
    return chunks

print(f"\n📐 Chunking {len(all_raw_text)} documents...")
ALL_CHUNKS = []
for source, text in all_raw_text.items():
    chunks = chunk_text(text, source)
    ALL_CHUNKS.extend(chunks)
    name = os.path.basename(source)
    print(f"   {name}: {len(chunks)} chunks")

print(f"\n   Total chunks: {len(ALL_CHUNKS)}")

# ── Build FAISS index ────────────────────────────────────────
print(f"\n🧠 Loading embedding model: {EMBED_MODEL} ...")
EMBED_MDL  = SentenceTransformer(EMBED_MODEL)

print(f"⚙️  Embedding {len(ALL_CHUNKS)} chunks...")
texts      = [c["text"] for c in ALL_CHUNKS]
embeddings = EMBED_MDL.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

dim         = embeddings.shape[1]
FAISS_INDEX = faiss.IndexFlatIP(dim)
FAISS_INDEX.add(embeddings)

# ── Sanity check ─────────────────────────────────────────────
test_vec = EMBED_MDL.encode(["staff management"], convert_to_numpy=True).astype("float32")
faiss.normalize_L2(test_vec)
scores, idxs = FAISS_INDEX.search(test_vec, 1)
if idxs[0][0] == -1:
    print("⚠️  WARNING: Index empty!")
else:
    print(f"\n   ✅ Index working — score: {scores[0][0]:.3f}")
    print(f"   Sample: {ALL_CHUNKS[idxs[0][0]]['text'][:100]}...")

print(f"""
🎉 RAG system ready!
   Files   : {len(all_raw_text)}
   Chunks  : {len(ALL_CHUNKS)}
   Vectors : {FAISS_INDEX.ntotal}
""")

📄 Converting 1 files with Docling...

   🔄 /content/sample_data/hivestaff_docs.pdf


[INFO] 2026-05-08 10:57:31,285 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-08 10:57:31,291 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-08 10:57:31,364 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-08 10:57:31,365 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-08 10:57:32,023 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-08 10:57:32,025 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-08 10:57:32,029 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-08 10:57:32,030 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-05-08 10:57:55,416 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-08 10:58:22,057 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-08 10:58:24,380 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-08 10:58:25,103 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-08 10:59:23,778 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-08 10:59:24,895 [RapidOCR] main.py:132: The text detection result is empty


      ✅ Docling: 59,004 chars

💾 Markdown saved to /content/docs_markdown/ (inspect if needed)

📐 Chunking 1 documents...
   hivestaff_docs.pdf: 185 chunks

   Total chunks: 185

🧠 Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⚙️  Embedding 185 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


   ✅ Index working — score: 0.571
   Sample: ## Shift   Roster

Create and manage employee shift schedules efficiently

<!-- image -->

## Leave
...

🎉 RAG system ready!
   Files   : 1
   Chunks  : 185
   Vectors : 185



In [5]:
# ============================================================
# CELL 3: Load Whisper Speech-to-Text Model
# ============================================================
import whisper

print(f"⏳ Loading Whisper model: {WHISPER_MODEL} ...")
whisper_model = whisper.load_model(WHISPER_MODEL)
print(f"✅ Whisper '{WHISPER_MODEL}' model loaded.")

⏳ Loading Whisper model: base ...
✅ Whisper 'base' model loaded.


In [6]:
# ============================================================
# CELL 4: LLaMA 3 via Groq — Real RAG + Emotion-Aware
# ============================================================
from groq import Groq

groq_client          = Groq(api_key=GROQ_API_KEY)
conversation_history = {}

USER_EMOTION_MAP = {
    "angry": [
        "unacceptable", "frustrated", "furious", "terrible", "worst",
        "ridiculous", "useless", "demand", "immediately", "angry",
        "so angry", "very angry", "not working", "fed up", "pathetic",
        "waste", "rubbish", "nonsense", "cheating", "lied", "bad service",
        "customer care", "complaint", "hate", "disgusting"
    ],
    "sad": [
        "sad", "upset", "disappointed", "unhappy", "depressed",
        "crying", "horrible", "awful", "devastated", "heartbroken",
        "not happy", "very sad", "feel bad"
    ],
    "happy": [
        "happy", "love", "amazing", "wonderful", "great", "fantastic",
        "excellent", "best", "thank you so much", "awesome", "perfect",
        "very good", "so good", "nice", "brilliant", "superb", "satisfied"
    ],
    "excited": [
        "excited", "thrilled", "can't wait", "incredible", "wow",
        "unbelievable", "so excited", "really excited"
    ],
    "fearful": [
        "worried", "scared", "afraid", "anxious", "concerned",
        "serious", "urgent", "problem", "issue", "error", "broken",
        "not working", "stopped working", "crash", "bug", "help me",
        "stuck", "confused", "don't know", "lost", "can't find",
        "unable to", "how do i", "how to"
    ],
    "surprised": [
        "shocked", "unexpected", "didn't expect", "surprised",
        "no way", "really", "wait", "seriously", "are you sure"
    ],
}

RESPONSE_EMOTION = {
    "angry":     "sad",
    "sad":       "sad",
    "happy":     "happy",
    "excited":   "excited",
    "fearful":   "calm",
    "surprised": "surprised",
    "neutral":   "neutral",
}

EMOTION_SYSTEM_PROMPT = {
    "sad":      "The user seems upset. Respond with genuine empathy. Be soft and caring.",
    "happy":    "The user is happy. Be warm, positive and enthusiastic.",
    "excited":  "The user is excited. Match their energy. Be upbeat.",
    "calm":     "The user seems worried. Be calm, clear and reassuring.",
    "surprised":"The user seems surprised. Acknowledge it and be helpful.",
    "neutral":  "Respond naturally. Be professional and helpful.",
}

def detect_emotion(user_text):
    u = user_text.lower()
    for emotion in ["angry", "fearful", "sad", "happy", "excited", "surprised"]:
        keywords = USER_EMOTION_MAP[emotion]
        if any(k in u for k in keywords):
            ai_emotion = RESPONSE_EMOTION[emotion]
            print(f"   → User emotion: {emotion} → AI responds as: {ai_emotion}")
            return ai_emotion
    print("   → neutral")
    return "neutral"


def retrieve_context(query: str, top_k: int = TOP_K_CHUNKS) -> str:
    query_vec = EMBED_MDL.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vec)

    scores, indices = FAISS_INDEX.search(query_vec, top_k)
    retrieved    = []
    seen_sources = set()

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        if score < 0.20:
            continue
        chunk        = ALL_CHUNKS[idx]
        source       = chunk["source"]
        text         = chunk["text"]
        source_label = f"[Source: {source}]\n" if source not in seen_sources else ""
        seen_sources.add(source)
        retrieved.append(f"{source_label}{text}")
        print(f"   📎 score={score:.3f}: {text[:60]}...")

    if not retrieved:
        return "NO_RELEVANT_CONTEXT"

    return "\n\n---\n\n".join(retrieved)


def get_ai_response(session_id: str, user_text: str) -> tuple[str, str]:

    if session_id not in conversation_history:
        conversation_history[session_id] = []

    detected_emotion    = detect_emotion(user_text)
    emotion_instruction = EMOTION_SYSTEM_PROMPT.get(
        detected_emotion, EMOTION_SYSTEM_PROMPT["neutral"]
    )

    print(f"\n🔍 RAG retrieval for: '{user_text[:60]}'")
    context = retrieve_context(user_text, top_k=TOP_K_CHUNKS)

    context_block = (
        "No relevant documentation found."
        if context == "NO_RELEVANT_CONTEXT"
        else context
    )

    system_prompt = f"""You are a friendly, natural-sounding voice assistant for HiveStaff on a phone call.
Speak like a confident, warm human agent — NOT a robot.
Keep answers to 1-2 SHORT sentences only. Be direct and conversational.
Never use lists, bullet points, numbers, or special characters.
Use natural speech patterns — contractions are fine (e.g. "you'll", "it's", "we've").

STRICT RULES:
1. Answer ONLY using the documentation context below.
2. If context doesn't have the answer say: "I don't have that detail, but our support team can help you with that."
3. Never invent features or steps not in the documentation.
4. If user speech is unclear, politely ask them to repeat.
5. {emotion_instruction}

=== HIVESTAFF DOCUMENTATION ===
{context_block}
=== END ===
"""

    messages = (
        [{"role": "system", "content": system_prompt}]
        + conversation_history[session_id]
        + [{"role": "user", "content": user_text}]
    )

    response = groq_client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=messages,
        max_tokens=80,       # ← reduced from 150 (shorter = faster TTS + more natural)
        temperature=0.4,
    )

    ai_reply = response.choices[0].message.content.strip()

    conversation_history[session_id].append({"role": "user",     "content": user_text})
    conversation_history[session_id].append({"role": "assistant", "content": ai_reply})

    if len(conversation_history[session_id]) > 16:
        conversation_history[session_id] = conversation_history[session_id][-16:]

    return ai_reply, detected_emotion


reply, emotion = get_ai_response("test", "I am so angry from your customer care service")
print(f"\nEmotion : {emotion}")
print(f"AI reply: {reply}")

   → User emotion: angry → AI responds as: sad

🔍 RAG retrieval for: 'I am so angry from your customer care service'
   📎 score=0.271: s
- Warning:

<!-- image -->

Deleted notices cannot be reco...
   📎 score=0.230: - [Contractor](https://document.hivestaff.in/modules/contrac...
   📎 score=0.221: -->

## Link   Documents

Connect GRN to purchase orders and...

Emotion : sad
AI reply: I'm so sorry to hear that you're feeling upset. That really concerns me, and I want to help you feel better. Can you please tell me what's going on and how I can assist you today?


In [7]:
# ============================================================
# CELL 5: Text-to-Speech — Natural Voices via Groq PlayAI
# ============================================================
import uuid, os, requests

AUDIO_DIR = "/tmp/audio_responses"
os.makedirs(AUDIO_DIR, exist_ok=True)

# ── Best natural-sounding PlayAI voices ──────────────────────
# Tested for phone clarity and natural tone
EMOTION_VOICE = {
    "sad":      "Aaliyah-PlayAI",   # warm, soft female
    "calm":     "Aaliyah-PlayAI",   # calm, reassuring
    "happy":    "Cole-PlayAI",      # natural, upbeat male
    "excited":  "Cole-PlayAI",      # energetic
    "surprised":"Cole-PlayAI",
    "neutral":  "Cole-PlayAI",      # clear, professional
}

def text_to_speech(text: str, emotion: str = "neutral") -> str:
    filename = f"{uuid.uuid4().hex}.wav"
    filepath = os.path.join(AUDIO_DIR, filename)

    voice = EMOTION_VOICE.get(emotion, "Cole-PlayAI")

    # Clean text for more natural speech
    text = (text
        .replace("*", "")
        .replace("#", "")
        .replace("•", "")
        .replace("  ", " ")
        .strip()
    )

    try:
        response = requests.post(
            "https://api.groq.com/openai/v1/audio/speech",
            headers={
                "Authorization": f"Bearer {GROQ_API_KEY}",
                "Content-Type":  "application/json"
            },
            json={
                "model":           "playai-tts",
                "input":           text,
                "voice":           voice,
                "response_format": "wav",
                "speed":           1.1,    # ← slightly faster than default (more natural)
            },
            timeout=8
        )

        if response.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(response.content)
            print(f"   🔊 Groq TTS [{emotion}, {voice}]: {filename}")
            return filename
        else:
            print(f"   ⚠️  Groq TTS failed ({response.status_code}) — gTTS fallback")
            raise Exception("Groq TTS failed")

    except Exception:
        from gtts import gTTS
        filename = f"{uuid.uuid4().hex}.mp3"
        filepath = os.path.join(AUDIO_DIR, filename)
        tts = gTTS(text=text, lang="en", slow=False)
        tts.save(filepath)
        print(f"   🔊 gTTS fallback [{emotion}]: {filename}")
        return filename


# Pre-generate greeting so /voice serves it instantly (zero TTS delay on pickup)
GREETING_TEXT     = "Hi there! I'm your HiveStaff assistant. How can I help you today?"
GREETING_FILENAME = text_to_speech(GREETING_TEXT, "neutral")
print(f"✅ Greeting pre-generated: {GREETING_FILENAME}")

   ⚠️  Groq TTS failed (400) — gTTS fallback
   🔊 gTTS fallback [neutral]: 907343454d1048a893b1f54244c2132f.mp3
✅ Greeting pre-generated: 907343454d1048a893b1f54244c2132f.mp3


In [8]:
# ============================================================
# CELL 6.5: Real-Time Call Logger
# ============================================================
from datetime import datetime
from collections import defaultdict
import json

call_logs       = defaultdict(list)
active_sessions = {}
call_recordings = defaultdict(list)  # ← NEW: stores recording URLs per session

def log_event(session_id, role, text, emotion=None, extra=None):
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry = {
        "time":    timestamp,
        "role":    role,
        "text":    text,
        "emotion": emotion,
        "extra":   extra or {},
    }
    call_logs[session_id].append(entry)

    # ── Save recording URL if present ────────────────────────
    if extra and extra.get("recording_url"):
        call_recordings[session_id].append({
            "time": timestamp,
            "url":  extra["recording_url"]
        })

    COLORS = {
        "SYSTEM": "\033[94m",
        "USER":   "\033[92m",
        "AI":     "\033[93m",
        "ERROR":  "\033[91m",
    }
    RESET = "\033[0m"
    BOLD  = "\033[1m"
    color = COLORS.get(role, "")

    emotion_tag = f" [{emotion}]" if emotion else ""
    extra_str   = f" | {json.dumps(extra)}" if extra else ""

    print(
        f"{color}{BOLD}[{timestamp}] [{role}]{emotion_tag}{RESET}"
        f"{color} {text}{extra_str}{RESET}"
    )


def log_call_start(session_id, caller_number):
    active_sessions[session_id] = {
        "caller":     caller_number,
        "start_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "turns":      0,
    }
    print("\n" + "═"*60)
    print(f"  📞  CALL STARTED  |  {caller_number}  |  Session: {session_id}")
    print("═"*60)
    log_event(session_id, "SYSTEM", f"Call started from {caller_number}")


def log_call_end(session_id):
    session = active_sessions.get(session_id, {})
    turns   = session.get("turns", 0)
    print("═"*60)
    print(f"  📵  CALL ENDED  |  {turns} turns  |  Session: {session_id}")
    print("═"*60 + "\n")
    log_event(session_id, "SYSTEM", f"Call ended after {turns} turns")


def print_full_transcript(session_id):
    logs = call_logs.get(session_id, [])
    if not logs:
        print("No logs found for session:", session_id)
        return
    print("\n" + "═"*60)
    print(f"  📋  FULL TRANSCRIPT  |  Session: {session_id}")
    print("═"*60)
    for entry in logs:
        emotion_tag = f" [{entry['emotion']}]" if entry['emotion'] else ""
        print(f"  {entry['time']}  {entry['role']:<8}{emotion_tag:<12}  {entry['text']}")
    print("═"*60 + "\n")


def print_all_sessions():
    if not call_logs:
        print("No calls logged yet.")
        return
    print("\n" + "═"*60)
    print("  📊  ALL CALL SESSIONS")
    print("═"*60)
    for sid, logs in call_logs.items():
        meta  = active_sessions.get(sid, {})
        turns = sum(1 for l in logs if l['role'] == 'USER')
        print(f"  Session : {sid}")
        print(f"  Caller  : {meta.get('caller', 'unknown')}")
        print(f"  Started : {meta.get('start_time', 'unknown')}")
        print(f"  Turns   : {turns}")
        print(f"  Events  : {len(logs)}")
        print("  " + "-"*40)
    print("═"*60 + "\n")


print("✅ Logger ready.")

✅ Logger ready.


In [9]:
# ============================================================
# CELL 6: Flask Server — Full Conversation Recording + Logs
# ============================================================
from flask import Flask, request, Response, send_from_directory
from twilio.twiml.voice_response import VoiceResponse
import requests as req
import os

app = Flask(__name__)

call_level_recording_sids = set()


def transcribe_with_groq(recording_url: str, session_id: str) -> str:
    try:
        audio_response = req.get(
            recording_url + ".wav",
            auth=(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN),
            timeout=15
        )
        if audio_response.status_code != 200:
            print(f"   ⚠️  Audio download failed: {audio_response.status_code}")
            return ""

        audio_path = f"/tmp/recording_{session_id}.wav"
        with open(audio_path, "wb") as f:
            f.write(audio_response.content)

        file_size = os.path.getsize(audio_path)
        print(f"   🎵 Audio downloaded: {file_size} bytes")

        if file_size < 5000:
            print(f"   ⚠️  Audio too small — likely silence")
            return ""

        with open(audio_path, "rb") as audio_file:
            transcription = groq_client.audio.transcriptions.create(
                file=("recording.wav", audio_file),
                model="whisper-large-v3-turbo",   # ← faster than large-v3, same quality
                language="en",
                response_format="text",
                prompt=(
                    "HiveStaff customer support call. Topics: HR, payroll, "
                    "recruitment, attendance, scheduling, assets, warehouse, modules."
                )
            )

        text = transcription.strip() if isinstance(transcription, str) else transcription.text.strip()
        print(f"   🎤 Groq Whisper: '{text}'")

        word_count = len(text.split())
        if word_count <= 2 and text.lower() not in [
            "yes", "no", "okay", "ok", "hello", "hi", "bye", "thank you", "thanks"
        ]:
            print(f"   ⚠️  Too short ({word_count} words) — unclear")
            return "UNCLEAR"

        return text

    except Exception as e:
        print(f"   ❌ Groq Whisper failed: {e}")
        return ""


@app.route("/voice", methods=["GET", "POST"])
def voice():
    response   = VoiceResponse()
    caller     = request.form.get("From", "unknown")
    call_sid   = request.form.get("CallSid", "")
    session_id = caller.replace("+", "").replace("-", "")

    if session_id not in active_sessions:
        log_call_start(session_id, caller)

    # ── Start full call recording via API ────────────────────
    # Pass callback URL explicitly so Twilio knows where to send completion
    if call_sid:
        try:
            rec = twilio_client.calls(call_sid).recordings.create(
                recording_status_callback=f"{PUBLIC_URL}/recording_status",
                recording_status_callback_method="POST",
            )
            call_level_recording_sids.add(rec.sid)
            call_recordings[session_id].append({
                "time": datetime.now().strftime("%H:%M:%S"),
                "url":  "",
                "sid":  rec.sid,
                "type": "full_call",
            })
            print(f"   🎙️ Full call recording started: {rec.sid}")
        except Exception as e:
            print(f"   ⚠️ Could not start call recording: {e}")

    # ── Play pre-generated greeting (instant — no TTS delay) ─
    log_event(session_id, "AI", GREETING_TEXT, emotion="neutral",
              extra={"type": "greeting"})

    greeting_url = f"{PUBLIC_URL}/audio/{GREETING_FILENAME}"
    response.play(greeting_url)

    response.record(
        action="/process_speech",
        method="POST",
        max_length=RECORD_MAX_LENGTH,
        timeout=RECORD_TIMEOUT,
        play_beep=False,
        recording_status_callback=f"{PUBLIC_URL}/recording_status",
    )
    return Response(str(response), mimetype="text/xml")


@app.route("/process_speech", methods=["POST"])
def process_speech():
    response      = VoiceResponse()
    caller        = request.form.get("From", "unknown")
    session_id    = caller.replace("+", "").replace("-", "")
    call_sid      = request.form.get("CallSid", "")
    recording_url = request.form.get("RecordingUrl", "")
    twilio_text   = request.form.get("SpeechResult", "").strip()

    log_event(session_id, "SYSTEM", "Webhook received", extra={
        "CallSid":      call_sid,
        "RecordingUrl": recording_url[:60] if recording_url else "",
    })

    user_text = ""
    if recording_url:
        user_text = transcribe_with_groq(recording_url, session_id)
    if not user_text and twilio_text:
        user_text = twilio_text
        print(f"   🔄 Twilio fallback: '{user_text}'")

    if user_text == "UNCLEAR" or not user_text:
        log_event(session_id, "SYSTEM", "Speech unclear — asking to repeat")
        unclear_audio = text_to_speech(
            "Sorry, I didn't catch that. Could you say it again?", "neutral"
        )
        response.play(f"{PUBLIC_URL}/audio/{unclear_audio}")
        response.record(
            action="/process_speech",
            method="POST",
            max_length=RECORD_MAX_LENGTH,
            timeout=RECORD_TIMEOUT,
            play_beep=False,
            recording_status_callback=f"{PUBLIC_URL}/recording_status",
        )
        return Response(str(response), mimetype="text/xml")

    log_event(session_id, "USER", user_text, extra={
        "source": "groq_whisper" if recording_url else "twilio_stt"
    })

    ai_reply, emotion = get_ai_response(session_id, user_text)
    log_event(session_id, "AI", ai_reply, emotion=emotion)

    if session_id in active_sessions:
        active_sessions[session_id]["turns"] = \
            active_sessions[session_id].get("turns", 0) + 1

    audio_filename = text_to_speech(ai_reply, emotion)
    audio_url      = f"{PUBLIC_URL}/audio/{audio_filename}"

    log_event(session_id, "SYSTEM", "Audio generated", extra={
        "file": audio_filename, "emotion": emotion
    })

    response.play(audio_url)
    response.record(
        action="/process_speech",
        method="POST",
        max_length=RECORD_MAX_LENGTH,
        timeout=RECORD_TIMEOUT,
        play_beep=False,
        recording_status_callback=f"{PUBLIC_URL}/recording_status",
    )
    return Response(str(response), mimetype="text/xml")


@app.route("/recording_status", methods=["POST"])
def recording_status():
    status        = request.form.get("RecordingStatus", "")
    recording_url = request.form.get("RecordingUrl", "")
    call_sid      = request.form.get("CallSid", "")
    recording_sid = request.form.get("RecordingSid", "")

    print(f"   📼 Recording status: {status} | SID: {recording_sid}")

    # Skip STT chunk recordings — only process full call recordings
    if recording_sid not in call_level_recording_sids:
        print(f"   ⏭️  STT chunk — skipped")
        return Response("OK", status=200)

    if status == "completed" and recording_url:
        try:
            local_filename = f"rec_{recording_sid}.wav"
            local_path     = os.path.join(AUDIO_DIR, local_filename)

            if os.path.exists(local_path):
                print(f"   ⏭️  Already downloaded: {local_filename}")
                return Response("OK", status=200)

            r = req.get(
                recording_url + ".wav",
                auth=(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN),
                timeout=30
            )
            if r.status_code == 200:
                with open(local_path, "wb") as f:
                    f.write(r.content)

                local_url = f"{PUBLIC_URL}/audio/{local_filename}"
                print(f"   ✅ Full call recording saved: {local_filename}")

                for sid, recs in call_recordings.items():
                    for rec in recs:
                        if rec.get("sid") == recording_sid:
                            rec["url"] = local_url
                            print(f"   🔁 URL updated for session: {sid}")
            else:
                print(f"   ⚠️  Download failed: {r.status_code}")

        except Exception as e:
            print(f"   ❌ Error saving recording: {e}")

    return Response("OK", status=200)


@app.route("/audio/<filename>")
def serve_audio(filename):
    return send_from_directory(AUDIO_DIR, filename)


@app.route("/status")
def status():
    return {
        "status":      "running",
        "stt":         "groq/whisper-large-v3-turbo",
        "llm":         LLAMA_MODEL,
        "sessions":    len(call_logs),
        "total_turns": sum(s.get("turns", 0) for s in active_sessions.values()),
    }


@app.route("/logs")
def logs_endpoint():
    EMOTION_COLORS = {
        "sad":      "#ef4444",
        "calm":     "#3b82f6",
        "happy":    "#22c55e",
        "excited":  "#f59e0b",
        "surprised":"#a855f7",
        "neutral":  "#6b7280",
    }
    ROLE_COLORS = {
        "USER":   "#1d4ed8",
        "AI":     "#15803d",
        "SYSTEM": "#6b7280",
        "ERROR":  "#dc2626",
    }

    sessions_html = ""
    if not call_logs:
        sessions_html = "<p style='color:#6b7280;text-align:center;padding:40px'>No calls yet.</p>"
    else:
        for sid, logs in call_logs.items():
            meta       = active_sessions.get(sid, {})
            turns      = sum(1 for l in logs if l['role'] == 'USER')
            recordings = [
                r for r in call_recordings.get(sid, [])
                if r.get("type") == "full_call"
            ]

            rec_html = ""
            if recordings:
                rec_items = ""
                for i, rec in enumerate(recordings):
                    url = rec.get("url", "")
                    if not url:
                        rec_items += f"""
                        <div style="display:flex;align-items:center;gap:12px;
                                    padding:10px 14px;background:#fef9c3;
                                    border-radius:8px;margin-bottom:8px">
                            <span style="color:#92400e;font-size:13px">
                                ⏳ Full Call Recording {i+1} — processing, refresh in a moment...
                            </span>
                        </div>"""
                    else:
                        rec_items += f"""
                        <div style="display:flex;align-items:center;gap:12px;
                                    padding:10px 14px;background:#f0fdf4;
                                    border-radius:8px;margin-bottom:8px">
                            <span style="color:#6b7280;font-size:12px;min-width:60px">
                                {rec['time']}
                            </span>
                            <span style="font-size:12px;color:#374151;min-width:160px;
                                         font-weight:600">
                                🎙️ Full Call Recording {i+1}
                            </span>
                            <audio controls style="height:34px;flex:1" src="{url}">
                                Your browser does not support audio.
                            </audio>
                        </div>"""

                rec_html = f"""
                <div style="margin-bottom:24px">
                    <h4 style="margin:0 0 10px;color:#374151;font-size:14px;font-weight:700">
                        🎙️ Full Call Recording — AI + User Both Sides ({len(recordings)})
                    </h4>
                    {rec_items}
                </div>"""

            rows = ""
            for entry in logs:
                role    = entry['role']
                emotion = entry.get('emotion', '')
                text    = entry['text']
                time    = entry['time']

                if role == "SYSTEM":
                    continue

                role_color    = ROLE_COLORS.get(role, "#6b7280")
                emotion_badge = ""
                if emotion:
                    ec = EMOTION_COLORS.get(emotion, "#6b7280")
                    emotion_badge = f"""
                    <span style="background:{ec};color:white;padding:2px 8px;
                                 border-radius:12px;font-size:11px;font-weight:600">
                        {emotion}
                    </span>"""

                row_bg = "#f0fdf4" if role == "AI" else "#eff6ff"
                rows += f"""
                <tr style="background:{row_bg};border-bottom:1px solid #e5e7eb">
                    <td style="padding:10px 12px;color:#6b7280;font-size:12px;
                               white-space:nowrap">{time}</td>
                    <td style="padding:10px 12px">
                        <span style="color:{role_color};font-weight:700;
                                     font-size:12px">{role}</span>
                    </td>
                    <td style="padding:10px 12px">{emotion_badge}</td>
                    <td style="padding:10px 12px;color:#111827;font-size:14px;
                               line-height:1.5">{text}</td>
                </tr>"""

            sessions_html += f"""
            <div style="background:white;border-radius:12px;
                        box-shadow:0 1px 3px rgba(0,0,0,0.1);
                        margin-bottom:24px;overflow:hidden">
                <div style="background:linear-gradient(135deg,#1e40af,#3b82f6);
                            padding:16px 20px;color:white">
                    <div style="display:flex;justify-content:space-between;
                                align-items:center;flex-wrap:wrap;gap:8px">
                        <div>
                            <div style="font-size:18px;font-weight:700">
                                📞 {meta.get('caller', sid)}
                            </div>
                            <div style="font-size:12px;opacity:0.85;margin-top:2px">
                                Session: {sid}
                            </div>
                        </div>
                        <div style="display:flex;gap:12px">
                            <div style="text-align:center;background:rgba(255,255,255,0.15);
                                        padding:6px 14px;border-radius:8px">
                                <div style="font-size:20px;font-weight:700">{turns}</div>
                                <div style="font-size:11px;opacity:0.85">Turns</div>
                            </div>
                            <div style="text-align:center;background:rgba(255,255,255,0.15);
                                        padding:6px 14px;border-radius:8px">
                                <div style="font-size:20px;font-weight:700">
                                    {len(recordings)}
                                </div>
                                <div style="font-size:11px;opacity:0.85">Recordings</div>
                            </div>
                            <div style="text-align:center;background:rgba(255,255,255,0.15);
                                        padding:6px 14px;border-radius:8px">
                                <div style="font-size:13px;font-weight:600">
                                    {meta.get('start_time', 'N/A')}
                                </div>
                                <div style="font-size:11px;opacity:0.85">Started</div>
                            </div>
                        </div>
                    </div>
                </div>
                <div style="padding:20px">
                    {rec_html}
                    <h4 style="margin:0 0 10px;color:#374151;font-size:14px;font-weight:700">
                        💬 Conversation
                    </h4>
                    <div style="overflow-x:auto;border-radius:8px;border:1px solid #e5e7eb">
                        <table style="width:100%;border-collapse:collapse;font-family:sans-serif">
                            <thead>
                                <tr style="background:#f9fafb;border-bottom:2px solid #e5e7eb">
                                    <th style="padding:10px 12px;text-align:left;color:#6b7280;
                                               font-size:12px;font-weight:600;white-space:nowrap">
                                        TIME</th>
                                    <th style="padding:10px 12px;text-align:left;color:#6b7280;
                                               font-size:12px;font-weight:600">ROLE</th>
                                    <th style="padding:10px 12px;text-align:left;color:#6b7280;
                                               font-size:12px;font-weight:600">EMOTION</th>
                                    <th style="padding:10px 12px;text-align:left;color:#6b7280;
                                               font-size:12px;font-weight:600">MESSAGE</th>
                                </tr>
                            </thead>
                            <tbody>{rows}</tbody>
                        </table>
                    </div>
                </div>
            </div>"""

    html = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>HiveStaff AI Call Logs</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            background: #f1f5f9; min-height: 100vh;
        }}
        .header {{
            background: linear-gradient(135deg, #0f172a, #1e40af);
            color: white; padding: 20px 32px;
            display: flex; justify-content: space-between;
            align-items: center; flex-wrap: wrap; gap: 12px;
        }}
        .stat-pill {{
            background: rgba(255,255,255,0.12); padding: 8px 16px;
            border-radius: 20px; font-size: 13px; font-weight: 600;
        }}
        .container {{ max-width: 1100px; margin: 0 auto; padding: 24px 16px; }}
        .refresh-btn {{
            background: #3b82f6; color: white; border: none;
            padding: 8px 18px; border-radius: 8px;
            cursor: pointer; font-size: 13px; font-weight: 600;
        }}
        .refresh-btn:hover {{ background: #2563eb; }}
    </style>
</head>
<body>
    <div class="header">
        <div>
            <div style="font-size:22px;font-weight:800">🤖 HiveStaff AI — Call Logs</div>
            <div style="font-size:13px;opacity:0.7;margin-top:4px">
                Live call monitoring dashboard
            </div>
        </div>
        <div style="display:flex;gap:10px;align-items:center;flex-wrap:wrap">
            <span class="stat-pill">📞 {len(call_logs)} Sessions</span>
            <span class="stat-pill">
                💬 {sum(s.get('turns',0) for s in active_sessions.values())} Total Turns
            </span>
            <button class="refresh-btn" onclick="location.reload()">🔄 Refresh</button>
        </div>
    </div>
    <div class="container">{sessions_html}</div>
</body>
</html>"""

    return html


print("✅ Flask ready — natural voice, fast STT, full call recording fixed.")

✅ Flask ready — natural voice, fast STT, full call recording fixed.


In [10]:
# ============================================================
# CELL 7: Kill old server + Start ngrok + Flask (fresh start)
# ============================================================
import os, threading
from pyngrok import ngrok, conf

# ── Kill any existing Flask on port 5000 ───────────────────
os.system("fuser -k 5000/tcp")
import time; time.sleep(1)
print("✅ Port 5000 cleared.")

# ── Kill old ngrok tunnels ──────────────────────────────────
ngrok.kill()

# ── Start fresh ngrok tunnel ────────────────────────────────
conf.get_default().auth_token = NGROK_AUTH_TOKEN
tunnel     = ngrok.connect(5000)
PUBLIC_URL = tunnel.public_url

print(f"""
╔══════════════════════════════════════════════════╗
║  🚀 Voice Agent is LIVE                          ║
╠══════════════════════════════════════════════════╣
║  Public URL : {PUBLIC_URL:<35} ║
║  Webhook    : {(PUBLIC_URL+'/voice'):<35} ║
╠══════════════════════════════════════════════════╣
║  👉 Paste Webhook URL into Twilio console        ║
╚══════════════════════════════════════════════════╝
""")

# ── Start Flask in background ───────────────────────────────
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
print("✅ Flask started.")

✅ Port 5000 cleared.

╔══════════════════════════════════════════════════╗
║  🚀 Voice Agent is LIVE                          ║
╠══════════════════════════════════════════════════╣
║  Public URL : https://sitcom-powdered-casino.ngrok-free.dev ║
║  Webhook    : https://sitcom-powdered-casino.ngrok-free.dev/voice ║
╠══════════════════════════════════════════════════╣
║  👉 Paste Webhook URL into Twilio console        ║
╚══════════════════════════════════════════════════╝

 * Serving Flask app '__main__'
✅ Flask started.
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


In [13]:
# ============================================================
# CELL 9: Outbound Call
# ============================================================
from twilio.rest import Client
import time

YOUR_PHONE_NUMBER = "+918866442269"

call = twilio_client.calls.create(
    to=YOUR_PHONE_NUMBER,
    from_=TWILIO_PHONE_NUMBER,
    url=f"{PUBLIC_URL}/voice",
    method="POST",
    # NO record= here — full recording is started inside /voice via API
)

print(f"✅ Outbound call placed!")
print(f"   Call SID : {call.sid}")
print(f"   Status   : {call.status}")

✅ Outbound call placed!
   Call SID : CA209128f406e780bce6eef380719e4c7e
   Status   : queued


In [14]:
# ============================================================
# CELL 10: View Logs — Run anytime to inspect calls
# ============================================================

# Show all sessions summary
print_all_sessions()

# Show full transcript for a specific session
# (replace with your actual session_id from logs above)
# print_full_transcript("9187XXXXXXXX")

# Or view in browser:
print(f"\n🌐 Browser log viewer: {PUBLIC_URL}/logs")


════════════════════════════════════════════════════════════
  📊  ALL CALL SESSIONS
════════════════════════════════════════════════════════════
  Session : 14057048906
  Caller  : +14057048906
  Started : 2026-05-08 11:00:03
  Turns   : 2
  Events  : 12
  ----------------------------------------
════════════════════════════════════════════════════════════


🌐 Browser log viewer: https://sitcom-powdered-casino.ngrok-free.dev/logs
